In [ ]:
import os
import csv
import json
import time
import requests

API_URL = "https://www.wikidata.org/w/api.php"
USER_AGENT = "SemTabThesisBot/1.0 (https://github.com/LeiY769; contact: leiyang677@gmail.com)"

CONFIG = {
    "data_dir": "Valid",
    "tables_dir": None,
    "gt_file": None,
    "out": "candidate_gen_train.json",
    "val_out": None,
    "val_ratio": 0.0,
    "language": "en",
    "cache": "wikidata_label_cache.json",
    "sleep": 0.1,
    "no_fetch": False,
    "max_aliases": 5,      
    "max_examples": 0,
}
SYSTEM_PROMPT = (
    "You are a candidate generation assistant for entity linking in tabular data. "
    "Given a cell value and its table context, generate a list of distinct Wikidata "
    "entity names that could match the cell value. Return ONLY the candidate names "
    "separated by semicolons, nothing else. Example output: Candidate1; Candidate2; Candidate3")
def _get(params, max_retries=5, sleep=0.1):
    headers = {"User-Agent": USER_AGENT}
    for attempt in range(max_retries):
        try:
            r = requests.get(API_URL, params={**params, "maxlag": 5}, headers=headers, timeout=30)
        except requests.RequestException:
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 429 or r.status_code >= 500:
            wait = float(r.headers.get("Retry-After", 2 ** attempt))
            time.sleep(wait)
            continue
        if r.status_code == 200:
            if sleep:
                time.sleep(sleep)
            return r.json()
        return None
    return None
def fetch_entities(qids, language="en", sleep=0.1):
    out = {}
    qids = [q for q in dict.fromkeys(qids) if q]
    for i in range(0, len(qids), 50):
        chunk = qids[i:i + 50]
        data = _get({"action": "wbgetentities", "ids": "|".join(chunk),"props": "labels|aliases","languages": f"{language}|en", "format": "json"}, sleep=sleep)
        if not data:
            continue
        for qid, entity in data.get("entities", {}).items():
            labels = entity.get("labels", {})
            lab = labels.get(language) or labels.get("en")
            aliases = entity.get("aliases", {}).get(language, [])
            out[qid] = {"label": lab["value"] if lab else "","aliases": [a["value"] for a in aliases]}
    return out
def load_cache(path):
    if path and os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}
def save_cache(cache, path):
    if not path:
        return
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)
def read_table(path):
    with open(path, "r", encoding="utf-8", newline="") as f:
        return list(csv.reader(f))
def cell_at(rows, r, c):
    if 0 <= r < len(rows) and 0 <= c < len(rows[r]):
        return str(rows[r][c]).strip()
    return ""
def clean_query(raw_query):
    query = str(raw_query).strip()
    query = query.replace('"', '').replace("**", "").strip()
    if query.endswith(".") and not str(raw_query).strip().endswith(".."):
        query = query[:-1].strip()
    return query
def build_context_string(rows, r, target_c, max_cells=5):
    if not (0 <= r < len(rows)):
        return ""
    vals = [str(v).strip() for c, v in enumerate(rows[r]) if c != target_c and str(v).strip() and str(v).strip().lower() != "nan"]
    if vals:
        return "row context: " + ", ".join(vals[:max_cells])
    return ""
def qid_from_url(url):
    return str(url).strip().rstrip("/").split("/")[-1]
def read_cea_gt(path):
    with open(path, "r", encoding="utf-8", newline="") as f:
        for parts in csv.reader(f):
            if len(parts) < 4:
                continue
            table_id, row, col, url = parts[0], parts[1], parts[2], parts[3]
            try:
                row, col = int(row), int(col)
            except ValueError:
                continue
            qid = qid_from_url(url)
            if qid.startswith("Q"):
                yield table_id, row, col, qid
def build_candidates_target(info, max_aliases=5):
    if not info:
        return ""
    names = []
    label = (info.get("label") or "").strip()
    if label:
        names.append(label)
    for alias in info.get("aliases", []):
        alias = (alias or "").strip()
        if alias and alias.lower() not in {n.lower() for n in names}:
            names.append(alias)
        if len(names) >= 1 + max_aliases:
            break
    return "; ".join(names)
def build_example(cell_value, context_str, candidates, qid):
    return {"system": SYSTEM_PROMPT,"cell_value": cell_value,"context": context_str,"candidates": candidates, "qid": qid,"table": None          }


def build_dataset(config=None):
    cfg = {**CONFIG, **(config or {})}

    tables_dir = cfg["tables_dir"] or os.path.join(cfg["data_dir"], "tables")
    gt_file = cfg["gt_file"] or os.path.join(cfg["data_dir"], "gt", "cea_gt.csv")

    if not os.path.isdir(tables_dir):
        raise FileNotFoundError(f"Tables folder not found: {tables_dir}")
    if not os.path.isfile(gt_file):
        raise FileNotFoundError(f"Ground truth not found: {gt_file}")

    gt = list(read_cea_gt(gt_file))
    if cfg["max_examples"]:
        gt = gt[:cfg["max_examples"]]
    print(f"Read {len(gt)} CEA ground-truth cells from {len({t for t, *_ in gt})} tables")

    table_cache = {}
    records = [] 
    missing_tables = set()
    for table_id, row, col, qid in gt:
        if table_id not in table_cache:
            tpath = os.path.join(tables_dir, table_id + ".csv")
            table_cache[table_id] = read_table(tpath) if os.path.exists(tpath) else None
        rows = table_cache[table_id]
        if rows is None:
            missing_tables.add(table_id)
            continue
        raw = cell_at(rows, row, col)
        cell_value = clean_query(raw)
        if not cell_value:
            continue
        context_str = build_context_string(rows, row, col)
        records.append((table_id, cell_value, context_str, qid))

    if missing_tables:
        print(f"warning:{len(missing_tables)} tables referenced in gt were not found on disk and skipped")
    print(f"Extracted {len(records)} usable cell mentions")
    info = {}
    if not cfg["no_fetch"]:
        cache = load_cache(cfg["cache"])
        needed = sorted({qid for *_, qid in records if qid not in cache})
        print(f"Wikidata: {len(cache)} cached, {len(needed)} to fetch")
        for i in range(0, len(needed), 50):
            batch = needed[i:i + 50]
            cache.update(fetch_entities(batch, cfg["language"], cfg["sleep"]))
            for q in batch:
                cache.setdefault(q, {"label": "", "aliases": []})
            if (i // 50) % 10 == 0:
                save_cache(cache, cfg["cache"])
                print(f"  fetched {min(i + 50, len(needed))}/{len(needed)}")
        save_cache(cache, cfg["cache"])
        info = cache

    examples = []
    skipped_no_label = 0
    for table_id, cell_value, context_str, qid in records:
        candidates = build_candidates_target(info.get(qid), cfg["max_aliases"])
        if not candidates:
            skipped_no_label += 1
            continue
        ex = build_example(cell_value, context_str, candidates, qid)
        ex["table"] = table_id
        examples.append(ex)
    if skipped_no_label:
        print(f"  warning: {skipped_no_label} cells skipped (no Wikidata label available)")
    print(f"Built {len(examples)} supervised examples")
    train, val = examples, []
    if cfg["val_out"] and cfg["val_ratio"] > 0:
        tables = sorted({e["table"] for e in examples})
        n_val = max(1, int(len(tables) * cfg["val_ratio"]))
        val_tables = set(tables[-n_val:])
        train = [e for e in examples if e["table"] not in val_tables]
        val = [e for e in examples if e["table"] in val_tables]

    with open(cfg["out"], "w", encoding="utf-8") as f:
        json.dump(train, f, ensure_ascii=False, indent=2)
    print(f"Wrote {len(train)} training examples -> {cfg['out']}")

    if cfg["val_out"] and val:
        with open(cfg["val_out"], "w", encoding="utf-8") as f:
            json.dump(val, f, ensure_ascii=False, indent=2)
        print(f"Wrote {len(val)} validation examples -> {cfg['val_out']}")

    if examples:
        print("sample example ")
        print(json.dumps(examples[0], ensure_ascii=False, indent=2))

    return train, val

In [ ]:
train, val = build_dataset({"data_dir": "Valid","out": "candidate_gen_train.json","val_out": "candidate_gen_valid.json","val_ratio": 0.1,"language": "en","cache": "wikidata_label_cache.json","max_aliases": 5})

In [ ]:
import json

def reconstruct_user(ex):
    user = f"Cell value: {ex['cell_value']}"
    if ex["context"]:
        user += f"\nContext: {ex['context']}"
    user += "\nCandidates:"
    return user

for name in ("candidate_gen_train.json", "candidate_gen_valid.json"):
    data = json.load(open(name, encoding="utf-8"))
    n_tables = len({e["table"] for e in data})
    avg_cand = sum(e["candidates"].count(";") + 1 for e in data) / max(len(data), 1)
    print(f"{name}: {len(data)} examples, {n_tables} tables, "f"{avg_cand:.1f} candidates/example on average")
tr = json.load(open("candidate_gen_train.json", encoding="utf-8"))
va = json.load(open("candidate_gen_valid.json", encoding="utf-8"))
assert not ({e["table"] for e in tr} & {e["table"] for e in va}), "Train/valid split error: tables appear in both sets"
print("Train/valid split OK, no table appears in both sets")
ex = va[0]
print("[SYSTEM]\n" + ex["system"])
print("\n[USER]\n" + reconstruct_user(ex))
print("\n[ASSISTANT / cible]\n" + ex["candidates"])

## Dataset synthetic data

From the wikidata take some random QID and adding some noises and effect on it to try to be able to train the LLM



In [ ]:
import random
import string
def _alpha_positions(s):
    return [i for i, ch in enumerate(s) if ch.isalpha()]
def _n_edits(n_chars, rng):
    return rng.randint(1, max(1, n_chars // 4))

def corrupt_delete(s, rng):
    pos = _alpha_positions(s)
    if not pos:
        return s
    k = min(_n_edits(len(pos), rng), len(pos) - 1) if len(pos) > 1 else 1
    chars = list(s)
    for i in sorted(rng.sample(pos, k), reverse=True):
        del chars[i]
    return "".join(chars)

def corrupt_insert(s, rng):
    if not s:
        return s
    k = _n_edits(len(s), rng)
    chars = list(s)
    for _ in range(k):
        chars.insert(rng.randint(0, len(chars)), rng.choice(string.ascii_lowercase))
    return "".join(chars)

def corrupt_substitute(s, rng):
    pos = _alpha_positions(s)
    if not pos:
        return s
    chars = list(s)
    for i in rng.sample(pos, _n_edits(len(pos), rng)):
        repl = rng.choice(string.ascii_lowercase)
        chars[i] = repl.upper() if chars[i].isupper() else repl
    return "".join(chars)

def corrupt_label(label, rng):
    r = rng.random()
    if r < 0.25:
        return label, "clean"
    if r < 0.50:
        return corrupt_delete(label, rng), "delete"
    if r < 0.75:
        return corrupt_insert(label, rng), "insert"
    return corrupt_substitute(label, rng), "substitute"
def build_synthetic_dataset(n_target=1000, qid_min=1000, qid_max=10_000_000,seed=0, language="en", max_aliases=5, val_ratio=0.1,cache="wikidata_synth_cache.json", sleep=0.1,out="candidate_gen_synth_train.json",val_out="candidate_gen_synth_valid.json"):
    rng = random.Random(seed)
    cache_data = load_cache(cache)

    collected = {}
    attempts, max_attempts = 0, n_target * 50
    while len(collected) < n_target and attempts < max_attempts:
        qids = [f"Q{rng.randint(qid_min, qid_max)}" for _ in range(50)]
        need = [q for q in qids if q not in cache_data]
        if need:
            cache_data.update(fetch_entities(need, language, sleep))
            for q in need:
                cache_data.setdefault(q, {"label": "", "aliases": []})
        for q in qids:
            info = cache_data.get(q)
            if info and info.get("label") and q not in collected:
                collected[q] = info
        attempts += 50
        if attempts % 500 == 0:
            print(f"  collected {len(collected)}/{n_target} (tried {attempts} QIDs)")
    save_cache(cache_data, cache)
    print(f"Collected {len(collected)} Wikidata entities with a label")

    items = list(collected.items())
    rng.shuffle(items)
    examples, mode_counts = [], {}
    for qid, info in items[:n_target]:
        label = info["label"]
        noisy, mode = corrupt_label(label, rng)
        if not noisy.strip():          
            noisy, mode = label, "clean"
        candidates = build_candidates_target(info, max_aliases)
        ex = build_example(noisy, "", candidates, qid)
        ex["table"] = "synthetic"
        ex["corruption"] = mode
        examples.append(ex)
        mode_counts[mode] = mode_counts.get(mode, 0) + 1

    print("Corruption mix:", {m: f"{c} ({c/len(examples):.0%})" for m, c in sorted(mode_counts.items())})

    n_val = max(1, int(len(examples) * val_ratio)) if val_out and val_ratio > 0 else 0
    val, train = examples[:n_val], examples[n_val:]

    with open(out, "w", encoding="utf-8") as f:
        json.dump(train, f, ensure_ascii=False, indent=2)
    print(f"Wrote {len(train)} training examples -> {out}")
    if n_val:
        with open(val_out, "w", encoding="utf-8") as f:
            json.dump(val, f, ensure_ascii=False, indent=2)
        print(f"Wrote {len(val)} validation examples -> {val_out}")
    for ex in examples[:6]:
        print(f"[{ex['corruption']:>10}] {ex['cell_value']!r:40} -> {ex['candidates']!r}")
    return train, val
synth_train, synth_val = build_synthetic_dataset(n_target=1000, qid_min=1000, qid_max=10_000_000, seed=0, val_ratio=0.1)